# AsyncIO in Python — Detailed Practical Tutorial

This notebook teaches Python `asyncio` from first principles.

You will learn:

1. What async programming means
2. Sync vs async execution
3. `async def`, `await`, and the event loop
4. `asyncio.gather()`
5. `asyncio.create_task()`
6. `asyncio.as_completed()`
7. Error handling
8. Timeouts
9. Rate limiting with `Semaphore`
10. Async API calls with `httpx`
11. Production-style async template
12. When not to use async

> Main idea: Async does not make one task faster. It makes waiting tasks overlap.

## 1. What is `asyncio`?

`asyncio` is Python's built-in library for writing concurrent programs.

It is very useful when your program spends time waiting for external systems, such as:

- API calls
- database queries
- file downloads
- web scraping
- LLM tool calls
- network services

It is not mainly designed for CPU-heavy work like:

- model training
- image processing
- large pandas transformations
- matrix multiplication
- heavy loops

For CPU-heavy work, multiprocessing, Spark, Ray, Dask, or GPU acceleration is usually better.

## 2. Sync vs Async Intuition

Let us first simulate API calls using normal synchronous Python.

Each task takes 2 seconds.

If we run 3 tasks one by one, total time will be around 6 seconds.

In [ ]:
import time

# Subroutines

def fetch_weather_sync(city):
    print(f"Fetching weather for {city}")
    time.sleep(2)
    return f"Weather data for {city}"

cities = ["Paris", "London", "Berlin"]

start = time.time()

for city in cities:
    result = fetch_weather_sync(city)
    print(result)

print("Total time:", round(time.time() - start, 2), "seconds")

Fetching weather for Paris
Weather data for Paris
Fetching weather for London
Weather data for London
Fetching weather for Berlin
Weather data for Berlin
Total time: 6.0 seconds


## 3. Async Version

Now we will write the same logic using `asyncio`.

Important difference:

- `time.sleep(2)` blocks everything.
- `await asyncio.sleep(2)` pauses only the current coroutine and allows other coroutines to run.

In Jupyter Notebook, we can directly use `await main()`.

In a normal `.py` file, use:

```python
asyncio.run(main())
```

In [ ]:
import asyncio
import time

# Coroutines

async def fetch_weather_async(city):
    print(f"Fetching weather for {city}")
    await asyncio.sleep(2)
    return f"Weather data for {city}"

async def main():
    cities = ["Paris", "London", "Berlin"]

    start = time.time()

    tasks = [fetch_weather_async(city) for city in cities]
    results = await asyncio.gather(*tasks)

    for result in results:
        print(result)

    print("Total time:", round(time.time() - start, 2), "seconds")

await main()

Fetching weather for Paris
Fetching weather for London
Fetching weather for Berlin
Weather data for Paris
Weather data for London
Weather data for Berlin
Total time: 2.0 seconds


### Key observation

The synchronous version took around 6 seconds.

The async version takes around 2 seconds.

Why?

Because all three waiting operations overlap.

## 4. `async def`

When you define a function using `async def`, it becomes a coroutine function.

Calling a coroutine function does not immediately execute it. It returns a coroutine object.

In [ ]:
def say_hello():
    return "Hello from sync function"

subroutine_object = say_hello()
print(subroutine_object)

Hello from sync function


In [ ]:
async def say_hello():
    return "Hello from async function"

coroutine_object = say_hello()
print(coroutine_object)

<coroutine object say_hello at 0x7debb2ebd850>


/tmp/ipykernel_2027/1063657433.py:4: RuntimeWarning: coroutine 'say_hello' was never awaited
  coroutine_object = say_hello()


To actually run the coroutine, you must `await` it.

In [ ]:
result = await say_hello()
print(result)

Hello from async function


## 5. `await`

`await` means:

> Pause here until this async operation completes. While waiting, allow the event loop to run other tasks.

Example:

In [ ]:
async def task(name):
    print(f"{name} started")
    await asyncio.sleep(2)
    print(f"{name} completed")
    return name

result = await task("Task A")
print("Result:", result)

Task A started
Task A completed
Result: Task A


## 6. The Event Loop

The event loop is the engine behind asyncio.

It keeps track of:

- which tasks are waiting
- which tasks are ready to continue
- which task should run next

In normal Python scripts, you usually start the event loop like this:

```python
asyncio.run(main())
```

In Jupyter notebooks, an event loop is already running, so we usually write:

```python
await main()
```

## 7. Sequential Async Execution

Async code can still be sequential if you await each task one by one.

The following example takes around 6 seconds.

In [ ]:
async def api_call(name):
    print(f"{name} started")
    await asyncio.sleep(2)
    print(f"{name} finished")
    return name

async def sequential_main():
    start = time.time()

    result1 = await api_call("Task 1")
    result2 = await api_call("Task 2")
    result3 = await api_call("Task 3")

    print(result1, result2, result3)
    print("Total:", round(time.time() - start, 2), "seconds")

await sequential_main()

Task 1 started
Task 1 finished
Task 2 started
Task 2 finished
Task 3 started
Task 3 finished
Task 1 Task 2 Task 3
Total: 6.01 seconds


## 8. Concurrent Execution with `asyncio.gather()`

`asyncio.gather()` runs multiple coroutines concurrently and returns the results as a list.

The result list preserves the input order, not the completion order.

In [ ]:
async def api_call(name):
    print(f"{name} started")
    await asyncio.sleep(2)
    print(f"{name} finished")
    return name

async def gather_main():
    start = time.time()

    results = await asyncio.gather(
        api_call("Task 1"),
        api_call("Task 2"),
        api_call("Task 3")
    )

    print("Results:", results)
    print("Total:", round(time.time() - start, 2), "seconds")

await gather_main()

Task 1 started
Task 2 started
Task 3 started
Task 1 finished
Task 2 finished
Task 3 finished
Results: ['Task 1', 'Task 2', 'Task 3']
Total: 2.0 seconds


## 9. `asyncio.create_task()`

`create_task()` schedules a coroutine to run soon in the event loop.

It is useful when you want to start some async work now and await it later.

In [ ]:
import asyncio
import time

async def fetch_data(id, sleep_time):
    print(f"Coroutine {id} starting to fetch data.")

    await asyncio.sleep(sleep_time)

    return {
        "id": id,
        "data": f"Sample data from coroutine {id}"
    }


async def main():
    start = time.time()

    # Create tasks for running coroutines concurrently
    task1 = asyncio.create_task(fetch_data(1, 2))
    task2 = asyncio.create_task(fetch_data(2, 10))
    task3 = asyncio.create_task(fetch_data(3, 1))

    # Await results
    result1 = await task1
    print(result1)
    result2 = await task2
    print(result2)
    result3 = await task3
    print(result3)

    print("\nFinal Results:")
    # print(result1, result2, result3)

    print("\nTotal time:", round(time.time() - start, 2), "seconds")

await main()

Coroutine 1 starting to fetch data.
Coroutine 2 starting to fetch data.
Coroutine 3 starting to fetch data.
{'id': 1, 'data': 'Sample data from coroutine 1'}
{'id': 2, 'data': 'Sample data from coroutine 2'}
{'id': 3, 'data': 'Sample data from coroutine 3'}

Final Results:

Total time: 10.01 seconds


### `gather()` vs `create_task()`

Use `gather()` when you have a batch of independent tasks and want all results.

Use `create_task()` when you want manual control over when a task starts and when you await it.

## 10. Simulating Multiple API Calls

Now let us simulate API calls with random delays.

This is similar to calling multiple services in an AI agent or backend application.

In [ ]:
import random

async def call_api(city):
    delay = random.randint(1, 4)
    print(f"Calling API for {city}, delay={delay}s")

    await asyncio.sleep(delay)

    return {
        "city": city,
        "temperature": random.randint(10, 35)
    }

async def api_batch_main():
    cities = ["Paris", "London", "Berlin", "Rome", "Madrid"]

    start = time.time()

    tasks = [call_api(city) for city in cities]
    results = await asyncio.gather(*tasks)

    print("\nFinal results:")
    for result in results:
        print(result)

    print("\nTotal time:", round(time.time() - start, 2), "seconds")

await api_batch_main()

Calling API for Paris, delay=1s
Calling API for London, delay=2s
Calling API for Berlin, delay=2s
Calling API for Rome, delay=1s
Calling API for Madrid, delay=1s

Final results:
{'city': 'Paris', 'temperature': 16}
{'city': 'London', 'temperature': 26}
{'city': 'Berlin', 'temperature': 34}
{'city': 'Rome', 'temperature': 21}
{'city': 'Madrid', 'temperature': 10}

Total time: 2.0 seconds


## 11. Handling Results as They Complete: `asyncio.as_completed()`

Sometimes you do not want to wait for all tasks before processing results.

`asyncio.as_completed()` gives results in completion order.

In [ ]:
async def call_api_with_delay(city):
    delay = random.randint(1, 5)
    await asyncio.sleep(delay)
    return f"{city} completed in {delay}s"

async def as_completed_main():
    cities = ["Paris", "London", "Berlin", "Rome", "Madrid"]

    tasks = [call_api_with_delay(city) for city in cities]

    for completed_task in asyncio.as_completed(tasks):
        result = await completed_task
        print(result)

await as_completed_main()

Madrid completed in 3s
London completed in 3s
Paris completed in 4s
Rome completed in 4s
Berlin completed in 5s


Useful when:

- you want to stream results
- you want progress updates
- you want to process the fastest responses first
- you are building scraping or agent pipelines

## 12. Error Handling in `asyncio.gather()`

By default, if one coroutine fails, `gather()` raises an exception.

In [ ]:
async def risky_task(name):
    await asyncio.sleep(1)

    if name == "B":
        raise ValueError("Task B failed")

    return f"{name} success"

async def failing_gather_main():
    try:
        results = await asyncio.gather(
            risky_task("A"),
            risky_task("B"),
            risky_task("C")
        )
        print(results)
    except Exception as e:
        print("Caught error:", repr(e))

await failing_gather_main()

If you want partial success, use:

```python
return_exceptions=True
```

In [ ]:
async def safe_gather_main():
    results = await asyncio.gather(
        risky_task("A"),
        risky_task("B"),
        risky_task("C"),
        return_exceptions=True
    )

    for result in results:
        if isinstance(result, Exception):
            print("Error:", result)
        else:
            print("Success:", result)

await safe_gather_main()

## 13. Timeouts

In production, never wait forever for an external service.

Use `asyncio.wait_for()`.

In [ ]:
async def slow_api():
    await asyncio.sleep(5)
    return "API response"

async def timeout_main():
    try:
        result = await asyncio.wait_for(slow_api(), timeout=2)
        print(result)
    except asyncio.TimeoutError:
        print("API timed out")

await timeout_main()

Timeouts are important for:

- LLM calls
- APIs
- databases
- microservices
- scraping

## 14. Rate Limiting with `asyncio.Semaphore`

If you fire too many requests at once, your API provider may throttle or block you.

A semaphore limits how many tasks can run at the same time.

In [ ]:
sem = asyncio.Semaphore(3)

async def limited_api_call(name):
    async with sem:
        print(f"Starting {name}")
        await asyncio.sleep(2)
        print(f"Finished {name}")
        return name

async def semaphore_main():
    tasks = [
        limited_api_call(f"Task {i}")
        for i in range(1, 11)
    ]

    start = time.time()
    results = await asyncio.gather(*tasks)

    print("Results:", results)
    print("Total time:", round(time.time() - start, 2), "seconds")

await semaphore_main()

Here, only 3 tasks run at once.

This is one of the most important production patterns in async programming.

## 15. AI Agent Example: Weather + Attractions

Imagine an agent has to process 5 cities.

For each city, it needs two independent tool calls:

1. Weather
2. Attractions

Total tool calls = 10

Sequential execution will be slow. Async execution can make these calls overlap.

In [ ]:
async def get_weather(city):
    await asyncio.sleep(2)
    return f"Weather for {city}"

async def get_attractions(city):
    await asyncio.sleep(2)
    return f"Attractions for {city}"

async def process_city(city):
    weather, attractions = await asyncio.gather(
        get_weather(city),
        get_attractions(city)
    )

    return {
        "city": city,
        "weather": weather,
        "attractions": attractions
    }

async def agent_main():
    cities = ["Paris", "London", "Berlin", "Rome", "Madrid"]

    start = time.time()

    results = await asyncio.gather(
        *[process_city(city) for city in cities]
    )

    for result in results:
        print(result)

    print("Total time:", round(time.time() - start, 2), "seconds")

await agent_main()

## 16. Same Agent Example with Rate Limit

Now we add a semaphore to avoid firing all tool calls at once.

In [ ]:
tool_semaphore = asyncio.Semaphore(3)

async def limited_tool_call(name):
    async with tool_semaphore:
        print(f"Starting {name}")
        await asyncio.sleep(2)
        print(f"Finished {name}")
        return name

async def get_weather_limited(city):
    return await limited_tool_call(f"weather:{city}")

async def get_attractions_limited(city):
    return await limited_tool_call(f"attractions:{city}")

async def process_city_limited(city):
    weather, attractions = await asyncio.gather(
        get_weather_limited(city),
        get_attractions_limited(city)
    )

    return {
        "city": city,
        "weather": weather,
        "attractions": attractions
    }

async def limited_agent_main():
    cities = ["Paris", "London", "Berlin", "Rome", "Madrid"]

    start = time.time()

    results = await asyncio.gather(
        *[process_city_limited(city) for city in cities]
    )

    print("\nFinal results:")
    for result in results:
        print(result)

    print("\nTotal time:", round(time.time() - start, 2), "seconds")

await limited_agent_main()

## 17. Async HTTP Calls with `httpx`

The normal `requests` library is synchronous.

For real async HTTP calls, you can use `httpx.AsyncClient`.

Install it if needed:

```bash
pip install httpx
```

The following cell is optional and requires internet access.

In [ ]:
# Optional example: run this only if your notebook has internet access.

# import httpx
# import asyncio
#
# async def fetch_url(client, url):
#     response = await client.get(url, timeout=10)
#     response.raise_for_status()
#     return response.text
#
# async def httpx_main():
#     urls = [
#         "https://example.com",
#         "https://example.org",
#         "https://example.net"
#     ]
#
#     async with httpx.AsyncClient() as client:
#         tasks = [fetch_url(client, url) for url in urls]
#         results = await asyncio.gather(*tasks)
#
#     for result in results:
#         print(result[:100])
#
# await httpx_main()

## 18. Production-Style Async Template

This example includes:

- concurrency limit
- timeout
- reusable HTTP client
- error handling
- clean return format

This is a good base structure for real projects.

In [ ]:
# Optional production-style template.
# Requires internet access and httpx package.

# import asyncio
# import httpx
# from typing import List, Dict, Any
#
#
# class APIClient:
#     def __init__(self, max_concurrency: int = 5, timeout: float = 10.0):
#         self.semaphore = asyncio.Semaphore(max_concurrency)
#         self.timeout = timeout
#
#     async def fetch_json(
#         self,
#         client: httpx.AsyncClient,
#         url: str
#     ) -> Dict[str, Any]:
#         async with self.semaphore:
#             try:
#                 response = await client.get(url, timeout=self.timeout)
#                 response.raise_for_status()
#                 return response.json()
#
#             except httpx.TimeoutException:
#                 return {
#                     "url": url,
#                     "error": "timeout"
#                 }
#
#             except httpx.HTTPStatusError as e:
#                 return {
#                     "url": url,
#                     "error": f"http_error_{e.response.status_code}"
#                 }
#
#             except Exception as e:
#                 return {
#                     "url": url,
#                     "error": str(e)
#                 }
#
#
# async def production_main():
#     urls = [
#         "https://jsonplaceholder.typicode.com/posts/1",
#         "https://jsonplaceholder.typicode.com/posts/2",
#         "https://jsonplaceholder.typicode.com/posts/3",
#         "https://jsonplaceholder.typicode.com/posts/4",
#         "https://jsonplaceholder.typicode.com/posts/5"
#     ]
#
#     api_client = APIClient(max_concurrency=3, timeout=5.0)
#
#     async with httpx.AsyncClient() as client:
#         tasks = [
#             api_client.fetch_json(client, url)
#             for url in urls
#         ]
#
#         results = await asyncio.gather(*tasks)
#
#     for result in results:
#         print(result)
#
#
# await production_main()

## 19. Common Mistakes

### Mistake 1: Forgetting `await`

Wrong:

```python
result = fetch_data()
```

Correct:

```python
result = await fetch_data()
```

---

### Mistake 2: Using `time.sleep()` inside async code

Wrong:

```python
time.sleep(2)
```

Correct:

```python
await asyncio.sleep(2)
```

---

### Mistake 3: Using `requests` inside async code

Wrong:

```python
import requests

async def fetch():
    response = requests.get(url)
```

Correct:

```python
import httpx

async def fetch(client, url):
    response = await client.get(url)
```

---

### Mistake 4: Calling `asyncio.run()` inside a notebook cell or another async function

In notebooks, prefer:

```python
await main()
```

In scripts, use:

```python
asyncio.run(main())
```

---

### Mistake 5: Parallelizing dependent tasks

Wrong:

```python
await asyncio.gather(
    get_user(),
    get_user_orders()
)
```

If orders require user ID, do this:

```python
user = await get_user()
orders = await get_user_orders(user["id"])
```

## 20. When Not to Use Async

Async is not the best solution for CPU-heavy work.

Bad fit:

- training ML models
- image processing
- big pandas operations
- local embedding generation
- heavy mathematical computation

Good fit:

- calling many APIs
- downloading many files
- scraping web pages
- running many DB queries
- calling many LLM tools
- websocket servers
- agentic workflows with independent tools

## 21. Interview-Ready Explanation

You can explain `asyncio` like this:

> `asyncio` is Python's concurrency framework for I/O-bound workloads. It uses an event loop to run coroutines. When a coroutine reaches `await`, it gives control back to the event loop so other tasks can continue. This is useful for APIs, database calls, file downloads, web scraping, and LLM tool calls. It does not make CPU-heavy code faster because the code is still mainly running on one thread unless we combine it with multiprocessing or executors.

## 22. Final Summary

AsyncIO in one line:

> Write code that can pause during waiting, let other tasks run, and resume later.

Important production checklist:

- Use `await` properly
- Avoid blocking calls like `time.sleep()` and `requests.get()`
- Use `asyncio.gather()` for independent tasks
- Use `asyncio.as_completed()` when you want results as they finish
- Use `Semaphore` for rate limits
- Use timeouts for external calls
- Handle partial failures
- Do not parallelize dependent steps

In [ ]:
import asyncio
import time

async def fetch_data(id, sleep_time):
    print(f"Coroutine {id} starting to fetch data.")

    await asyncio.sleep(sleep_time)

    return {
        "id": id,
        "data": f"Sample data from coroutine {id}"
    }


async def main():
    start = time.time()

    # Create tasks for running coroutines concurrently
    task1 = asyncio.create_task(fetch_data(1, 2))
    task2 = asyncio.create_task(fetch_data(2, 10))
    task3 = asyncio.create_task(fetch_data(3, 1))

    # Await results
    result1 = await task1
    print(result1)
    result2 = await task2
    print(result2)
    result3 = await task3
    print(result3)

    print("\nFinal Results:")
    # print(result1, result2, result3)

    print("\nTotal time:", round(time.time() - start, 2), "seconds")

await main()

Coroutine 1 starting to fetch data.
Coroutine 2 starting to fetch data.
Coroutine 3 starting to fetch data.
{'id': 1, 'data': 'Sample data from coroutine 1'}
{'id': 2, 'data': 'Sample data from coroutine 2'}
{'id': 3, 'data': 'Sample data from coroutine 3'}

Final Results:

Total time: 10.01 seconds
